In [2]:
import pandas as pd


In [4]:
oes = pd.read_excel('../houston_data/oesm24in4/nat3d_M2024_dl.xlsx')
exposure = pd.read_csv('../houston_data/job_exposure.csv')

In [5]:
oes_detail = oes[oes['O_GROUP'] == 'detailed'].copy()

In [6]:
oes_detail['TOT_EMP'] = pd.to_numeric(oes_detail['TOT_EMP'], errors='coerce')

In [7]:
merged = oes_detail.merge(exposure[['occ_code', 'observed_exposure']], 
                          left_on='OCC_CODE', right_on='occ_code', how='left')


In [8]:
def weighted_avg(group):
    valid = group.dropna(subset=['observed_exposure', 'TOT_EMP'])
    if valid['TOT_EMP'].sum() == 0:
        return None
    return (valid['observed_exposure'] * valid['TOT_EMP']).sum() / valid['TOT_EMP'].sum()

In [9]:

industry_exposure = (
    merged.groupby(['NAICS', 'NAICS_TITLE'])
    .apply(weighted_avg, include_groups=False)
    .reset_index(name='AI_Exposure_Score')
    .sort_values('AI_Exposure_Score', ascending=False)
)

In [10]:
industry_exposure.head(10)

,NAICS,NAICS_TITLE,AI_Exposure_Score
61,523000,"Securities, Commodity Contracts, and Other Fin...",0.354952
32,425000,Wholesale Trade Agents and Brokers,0.339953
57,518000,"Computing Infrastructure Providers, Data Proce...",0.328549
54,513000,Publishing Industries,0.301927
63,525000,"Funds, Trusts, and Other Financial Vehicles",0.296140
62,524000,Insurance Carriers and Related Activities,0.291291
66,533000,Lessors of Nonfinancial Intangible Assets (exc...,0.289182
60,522000,Credit Intermediation and Related Activities,0.274033
58,519000,"Web Search Portals, Libraries, Archives, and O...",0.273355
40,458000,"Clothing, Clothing Accessories, Shoe, and Jewe...",0.263430


In [11]:
industry_exposure['NAICS_2digit'] = industry_exposure['NAICS'].astype(str).str[:2]


In [12]:
sector_exposure = (
    industry_exposure.groupby('NAICS_2digit')['AI_Exposure_Score']
    .mean()
    .reset_index()
    .sort_values('AI_Exposure_Score', ascending=False)
)

In [14]:
houston = pd.read_excel('../houston_data/Qcew Report - Houston data level 3.xlsx')


In [15]:
industry_exposure['NAICS_3digit'] = industry_exposure['NAICS'].astype(str).str[:3]
houston['NAICS_3digit'] = houston['Industry Code'].astype(str).str[:3]

In [16]:
houston_merged = houston.merge(industry_exposure[['NAICS_3digit', 'AI_Exposure_Score']], 
                               on='NAICS_3digit', how='left')

In [17]:
print(houston_merged[['Industry', 'NAICS_3digit', 'AI_Exposure_Score']].drop_duplicates().sort_values('AI_Exposure_Score', ascending=False).head(15))
print(f"\n匹配成功: {houston_merged['AI_Exposure_Score'].notna().sum()} 行")
print(f"未匹配: {houston_merged['AI_Exposure_Score'].isna().sum()} 行")

                                               Industry NAICS_3digit  \
939   Securities, Commodity Contracts, and Other Fin...          523   
525                  Wholesale Trade Agents and Brokers          425   
886   Computing Infrastructure Providers, Data Proce...          518   
841                               Publishing Industries          513   
969        Funds, Trusts, and Other Financial Vehicles           525   
954           Insurance Carriers and Related Activities          524   
1014  Lessors of Nonfinancial Intangible Assets (exc...          533   
924        Credit Intermediation and Related Activities          522   
901   Web Search Portals, Libraries, Archives, and O...          519   
645   Clothing, Clothing Accessories, Shoe, and Jewe...          458   
1044            Management of Companies and Enterprises          551   
916                   Monetary Authorities-Central Bank          521   
1029   Professional, Scientific, and Technical Services         

In [18]:
unmatched = houston_merged[houston_merged['AI_Exposure_Score'].isna()][['Industry', 'NAICS_3digit']].drop_duplicates()
print(unmatched)

                               Industry NAICS_3digit
0                       Crop Production          111
15    Animal Production and Aquaculture          112
45        Fishing, Hunting and Trapping          114
1284                 Private Households          814
